
# DataStream Class
This tutorial demonstrates the usage of the DataStream class,
which provides methods for analyzing time-series data.

The following features are:
    - **Trimming**: Identifies steady-state regions in data.
    - **Statistical Analysis**: Computes mean, standard deviation, confidence intervals, and cumulative statistics.
    - **Stationarity Testing**: Uses the Augmented Dickey-Fuller test.
    - **Effective Sample Size (ESS)**: Estimates the independent sample size.
    - **Optimal Window Size**: Determines the best window for data smoothing.


Import DataStream



In [ ]:
import quends as qnds

# GX Data Analysis
Analysis on GX Data



In [ ]:
# Specify the file paths
csv_file_path = "gx/tprim_2_0.out.csv"
csv2_file_path = "gx/ensemble/tprim_2_5_a.out.csv"

# Load the data from CSV files
data_stream_csv = qnds.from_csv(csv_file_path, "HeatFlux_st")
data_stream_gx = qnds.from_csv(csv2_file_path, "HeatFlux_st")

# Display the first few rows of the GX data
data_stream_gx.head()

Get available variables



In [ ]:
data_stream_gx.variables()

Get number of rows from the following data in GX



In [ ]:
len(data_stream_gx)

## Stationary Check




In [ ]:
# Check if a single column is stationary
data_stream_gx.is_stationary("HeatFlux_st")

# Check stationarity for several variables. With the single-variable API,
# each column is loaded into its own DataStream.
for _var in ["HeatFlux_st", "Wg_st", "Phi2_t"]:
    print(_var, qnds.from_csv(csv2_file_path, _var).is_stationary(_var))

## Trimming data based to obtain steady-state portion




Trim the data based on standard deviation method (Quantile strategy)
Use the strategy-operation pattern from quends.base.trim directly.



In [ ]:
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

strategy = QuantileTrimStrategy(window_size=50, robust=True)
op = TrimDataStreamOperation(strategy=strategy)
trimmed = op(data_stream_gx, column_name="HeatFlux_st")

# Print first 5 rows of dataframe
trimmed.head()

Trim the data based on rolling variance method



In [ ]:
from quends.base.trim import RollingVarianceThresholdTrimStrategy, TrimDataStreamOperation

strategy = RollingVarianceThresholdTrimStrategy(window_size=50, threshold=0.10)
op = TrimDataStreamOperation(strategy=strategy)
trimmed = op(data_stream_gx, column_name="HeatFlux_st")

# Gather results
trimmed.head()

Trim the data based on noise threshold method



In [ ]:
from quends.base.trim import NoiseThresholdTrimStrategy, TrimDataStreamOperation

strategy = NoiseThresholdTrimStrategy(window_size=50, threshold=0.1)
op = TrimDataStreamOperation(strategy=strategy)
trimmed = op(data_stream_gx, column_name="HeatFlux_st")

# View trimmed data
trimmed.head()

## Effective Sample Size

Compute Effective Sample Size for specific columns in GX. With the
single-variable API, each column is loaded into its own DataStream.



In [ ]:
for _var in ["HeatFlux_st", "Wg_st"]:
    print(_var, qnds.from_csv(csv2_file_path, _var).effective_sample_size())

Compute Effective sample size for trimmed data



In [ ]:
ess_df = trimmed.effective_sample_size()
print(ess_df)

# UQ Analysis

Compute Statistics on trimmed dataframe



In [ ]:
stats = trimmed.compute_statistics(method="sliding")
print(stats)

stats_df = stats["HeatFlux_st"]

Exporter
Below Displays the information as a DataFrame



In [ ]:
exporter = qnds.Exporter()
exporter.display_dataframe(stats)

Below Displays the information in JSON



In [ ]:
exporter.display_json(stats)

## Other statistical methods



Calculate the mean with a window size of 10



In [ ]:
mean_df = trimmed.mean(window_size=10)
print(mean_df)

Calculate the mean with the method of sliding



In [ ]:
mean_df = trimmed.mean(method="sliding")
print(mean_df)

Calculate the mean uncertainty



In [ ]:
uq_df = trimmed.mean_uncertainty()
print(uq_df)

Calculate the mean uncertainty with the method of sliding



In [ ]:
uq_df = trimmed.mean_uncertainty(method="sliding")
uq_df

Calculate the confidence intervale with the trimmed dataframe



In [ ]:
ci_df = trimmed.confidence_interval()
print(ci_df)

Cumlative Statistics



In [ ]:
cumulative = trimmed.cumulative_statistics()
print(cumulative)

cumulative_df = cumulative["HeatFlux_st"]

Display Cumulative Statistics as a DataFrame



In [ ]:
exporter.display_dataframe(cumulative)

## CGYRO Data Analysis




Specify the file paths



In [ ]:
csv_file_path = "cgyro/output_nu0_50.csv"
data_stream_cg = qnds.from_csv(csv_file_path, "Q_D/Q_GBD")
data_stream_cg.head()

Get the number of rows



In [ ]:
len(data_stream_cg)

Trim the CGYRO data using the Quantile (std) strategy



In [ ]:
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation
strategy = QuantileTrimStrategy(robust=True)
op = TrimDataStreamOperation(strategy=strategy)
trimmed_ = op(data_stream_cg, column_name="Q_D/Q_GBD")
# View trimmed data
print(trimmed_)

In [ ]:
trimmed_.head()

To check if data stream is stationary



In [ ]:
data_stream_cg.is_stationary("Q_D/Q_GBD")

To Plot for DataStream



In [ ]:
plotter = qnds.Plotter()
plot = plotter.trace_plot(data_stream_cg, ["Q_D/Q_GBD"])

In [ ]:
plot = plotter.steady_state_automatic_plot(
    data_stream_cg, variables_to_plot=["Q_D/Q_GBD"]
)

In [ ]:
plot = plotter.steady_state_plot(data_stream_cg, variables_to_plot=["Q_D/Q_GBD"])

To show additional data use:



In [ ]:
addition_info = trimmed.additional_data(method="sliding")
print(addition_info)

To add a reduction factor



In [ ]:
addition_info = trimmed.additional_data(reduction_factor=0.2)
print(addition_info)